# Building a Research Assistant with Web + X Search

What do reputable news sources say about a topic? What does the crowd on X think? And where do those two stories diverge?

In this guide, we'll build a research assistant that cross-references reporting from vetted news outlets with public discourse on X, then produces a structured "divergence briefing" that clusters claims into five categories: consensus, X-ahead-of-press, press-ahead-of-X, X-only, and press-only.

This takes advantage of Grok's built-in web search and X search as native tools, with domain filtering to scope web results to trusted outlets and inline citations linking every claim back to its source.

### What we'll use
- Web search with domain filtering (`allowed_domains`) to restrict results to trusted outlets
- X search to capture real-time public discourse
- Inline citations linking every claim back to its source
- Structured output with Pydantic models to parse the final analysis
- The native xai-sdk

### Table of Contents
- [Setup](#setup)
- [Act 1: Web Search with Domain Filtering](#act-1-web-search-with-domain-filtering)
- [Interlude: Filtered vs. Unfiltered](#interlude-filtered-vs-unfiltered)
- [Act 2: X Search for Public Discourse](#act-2-x-search-for-public-discourse)
- [Act 3: The Divergence Analysis](#act-3-the-divergence-analysis)
- [Structured Output](#structured-output)
- [Putting It All Together](#putting-it-all-together)
- [Conclusion](#conclusion)

## Setup

We use the native xai-sdk, which gives us direct access to Grok's search tools as first-class citizens, including domain filtering, X handle filtering, and inline citations.

All you need is an xAI API key.

In [1]:
%%capture
%pip install -q xai-sdk python-dotenv

In [2]:
import os

from dotenv import load_dotenv
from xai_sdk import Client
from xai_sdk.chat import system, user
from xai_sdk.tools import web_search, x_search

load_dotenv()

client = Client(api_key=os.getenv("XAI_API_KEY"))

MODEL = "grok-4.20-reasoning"

## Act 1: Web Search with Domain Filtering

Grok's `web_search` tool lets you restrict results to specific domains using `allowed_domains` (max 5 per call). Instead of hoping the model finds good sources, you tell it where to look.

We'll define curated domain lists for different research contexts, then run a search scoped to major news outlets.

In [3]:
# Curated domain lists for different research contexts
NEWS_DOMAINS = ["reuters.com", "apnews.com", "bbc.com", "ft.com", "wsj.com"]
TECH_DOMAINS = ["techcrunch.com", "arstechnica.com", "theverge.com", "wired.com"]
SCIENCE_DOMAINS = ["nature.com", "sciencedirect.com", "newscientist.com"]

In [4]:
TOPIC = "Space-based data centres and the future of orbital computing infrastructure"

chat = client.chat.create(
    model=MODEL,
    tools=[web_search(allowed_domains=NEWS_DOMAINS)],
    include=["inline_citations"],
)
chat.append(user(
    f"Search for the latest reporting on: {TOPIC}. "
    "Summarize the 3-5 most important claims from these sources, with citations. Be concise."
))

response = None
for response, chunk in chat.stream():
    if chunk.content:
        print(chunk.content, end="", flush=True)

**Key claims from recent reporting (primarily late 2025–early 2026):**

- **Space-based data centers offer major advantages for AI workloads**, including constant 24/7 solar power, the ability to radiate heat directly into the vacuum of space (eliminating water-intensive cooling), and bypassing Earth’s power-grid and land constraints amid surging AI demand. Musk has claimed space could become the lowest-cost location for AI computing “within two years, three at the latest.”[[1]](https://www.reuters.com/sustainability/climate-energy/why-does-elon-musk-want-put-ai-data-centers-space-2026-01-29/)[[2]](https://www.reuters.com/business/aerospace-defense/spacexs-orbital-data-centers-could-face-same-hurdles-microsofts-abandoned-2026-04-01/)

- **Ambitious deployment plans are underway from multiple players**: SpaceX is seeking approvals for up to 1 million solar-powered data-center satellites (with IPO funding), while Blue Origin (Project Sunrise), Starcloud (targeting an 88,000-satellite con

Every claim is backed by an inline citation linking to the original article. Let's inspect those citations programmatically:

In [5]:
print("Web search citations:")
for citation in response.inline_citations:
    if citation.HasField("web_citation"):
        print(f"  {citation.web_citation.url}")

Web search citations:
  https://www.reuters.com/sustainability/climate-energy/why-does-elon-musk-want-put-ai-data-centers-space-2026-01-29/
  https://www.reuters.com/business/aerospace-defense/spacexs-orbital-data-centers-could-face-same-hurdles-microsofts-abandoned-2026-04-01/
  https://www.reuters.com/business/aerospace-defense/spacexs-orbital-data-centers-could-face-same-hurdles-microsofts-abandoned-2026-04-01/
  https://www.reuters.com/sustainability/climate-energy/why-does-elon-musk-want-put-ai-data-centers-space-2026-01-29/
  https://www.bbc.com/news/articles/cjewvpkw7weo
  https://www.reuters.com/business/aerospace-defense/spacexs-orbital-data-centers-could-face-same-hurdles-microsofts-abandoned-2026-04-01/


Notice every source is from our `NEWS_DOMAINS` list.